# 03. Multi-Objective Optimization (NSGA-II) & Pareto Analysis

Runs NSGA-II over controllable kiln parameters (recomputing thermodynamic features inside every evaluation), extracts Low Energy, Balanced, and Low CO2 regimes, and displays setpoint recommendations.

In [ ]:
from cement_opt.config import load_config
from cement_opt.models import load_artifacts, build_validation_table
from cement_opt.data import load_and_merge_datasets, prepare_modeling_dataframe
from cement_opt.optimization import CementOptimizationProblem, ProcessOptimizer, run_nsga2

cfg = load_config("../configs/default.yaml")
merged = load_and_merge_datasets(cfg.paths.energy_csv, cfg.paths.co2_csv)
df = prepare_modeling_dataframe(merged, include_fuel_encoding=cfg.data.include_fuel_encoding)

energy_model, co2_model = load_artifacts(cfg.paths.models_dir)
problem = CementOptimizationProblem(energy_model, co2_model, production_column=cfg.data.production_column)
opt_result = run_nsga2(problem, cfg.optimization, cfg.thermodynamics)

optimizer = ProcessOptimizer(energy_model, co2_model, opt_result)
display(optimizer.get_representative_solutions())
display(optimizer.recommend_adjustments("Balanced"))

In [ ]:
balanced = optimizer.get_balanced_solution()
validation_table = build_validation_table(energy_model, co2_model, df, balanced)
display(validation_table)